In [1]:
import spacy as sp
import pandas as pd
import numpy


In [2]:
df=pd.read_csv("spam.csv")

In [3]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [4]:
df.drop(columns=["Unnamed: 2","Unnamed: 3","Unnamed: 4"],axis=1,inplace=True)

In [5]:
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
df.rename(columns={"v1":"target",
                   "v2":"text"},inplace=True)

In [7]:
df

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [8]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df["target"]=le.fit_transform(df["target"])

In [9]:
nlp=sp.load("en_core_web_sm")

In [33]:
def prep(text):
    text=text.lower()
    doc=nlp(text)
    tokens=[token.lemma_
        for token in doc 
        if not token.is_stop
        and not token.is_punct
    ]
    return " ".join(tokens)

In [36]:
df["text"]=df["text"].apply(prep)

In [37]:
df

,target,text
0,0,jurong point crazy available bugis n great wor...
1,0,ok lar joke wif u oni
2,1,free entry 2 wkly comp win fa cup final tkts 2...
3,0,u dun early hor u c
4,0,nah think go usf live
...,...,...
5567,1,2nd time try 2 contact u. u win å£750 pound pr...
5568,0,ì b go esplanade fr home
5569,0,pity mood suggestion
5570,0,guy bitching act like interested buy week give...


In [38]:
df.drop_duplicates(keep="first",inplace=True)

In [39]:
df

,target,text
0,0,jurong point crazy available bugis n great wor...
1,0,ok lar joke wif u oni
2,1,free entry 2 wkly comp win fa cup final tkts 2...
3,0,u dun early hor u c
4,0,nah think go usf live
...,...,...
5567,1,2nd time try 2 contact u. u win å£750 pound pr...
5568,0,ì b go esplanade fr home
5569,0,pity mood suggestion
5570,0,guy bitching act like interested buy week give...


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=500)

In [41]:
x=tf.fit_transform(df["text"]).toarray()
y=df["target"].values

In [42]:
from sklearn.model_selection import train_test_split
x_trian,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [47]:
svc=SVC(kernel="sigmoid",gamma=1)
lr=LogisticRegression(solver="liblinear")
mnb=MultinomialNB()
dtc=DecisionTreeClassifier(max_depth=5)
rfc=RandomForestClassifier(random_state=42,n_estimators=50)
abc=AdaBoostClassifier(random_state=42,n_estimators=50)
bc=BaggingClassifier(random_state=42,n_estimators=50)
etc=ExtraTreesClassifier(random_state=42,n_estimators=50)
gbc=GradientBoostingClassifier(random_state=42,n_estimators=50)
xgb=XGBClassifier(random_state=42,n_estimators=50)

In [48]:
cls={
    "svc":svc,
    "lr":lr,
    "mnb":mnb,
    "dtc":dtc,
    "rfc":rfc,
    "abc":abc,
    "bc":bc,
    "etc":etc,
    "gbc":gbc,
    "xgb":xgb
}

In [49]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
for name,model in cls.items():
    model.fit(x_trian,y_train)
    y_pred=model.predict(x_test)
    print(f"------{name}------")
    print("Accuracy:",accuracy_score(y_test,y_pred))
    print("Precision:",precision_score(y_test,y_pred))
    print("Recall:",recall_score(y_test,y_pred))
    print("F1 Score:",f1_score(y_test,y_pred))
    print("\n")

------svc------
Accuracy: 0.9666011787819253
Precision: 0.9541284403669725
Recall: 0.7819548872180451
F1 Score: 0.859504132231405


------lr------
Accuracy: 0.9646365422396856
Precision: 0.98989898989899
Recall: 0.7368421052631579
F1 Score: 0.8448275862068966


------mnb------
Accuracy: 0.9724950884086444
Precision: 0.972972972972973
Recall: 0.8120300751879699
F1 Score: 0.8852459016393442


------dtc------
Accuracy: 0.9332023575638507
Precision: 0.9452054794520548
Recall: 0.518796992481203
F1 Score: 0.6699029126213593


------rfc------
Accuracy: 0.9764243614931237
Precision: 0.990990990990991
Recall: 0.8270676691729323
F1 Score: 0.9016393442622951


------abc------
Accuracy: 0.9174852652259332
Precision: 0.9622641509433962
Recall: 0.38345864661654133
F1 Score: 0.5483870967741935


------bc------
Accuracy: 0.9705304518664047
Precision: 0.9401709401709402
Recall: 0.8270676691729323
F1 Score: 0.88


------etc------
Accuracy: 0.9774066797642437
Precision: 0.9824561403508771
Recall: 0.84210